In [1]:
from utils import *
from CS_feature_extractor import *
from CS_based_early_stopping import *

[nltk_data] Downloading package punkt to /home/guangya/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/guangya/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# How to run the code to get Acc and # of API calls

In [2]:
DATA_DIR = "../data/Evaluation_CoTs/"
input_file_path = os.path.join(DATA_DIR, 'GSM8K_GPT4o_mini.csv')
df_llama = pd.read_csv(input_file_path).reset_index(drop=True)

In [3]:
df_test = df_llama.sample(100).reset_index(drop=True) # get partial data for test

In [4]:
df_test  # Please put your question and CoT in the given format

,Name,Category,Question,Correct Answer,CoT_0,Final Answer_0,Instruction Violation_0,CoT_1,Final Answer_1,Instruction Violation_1,...,CoT_37,Final Answer_37,Instruction Violation_37,CoT_38,Final Answer_38,Instruction Violation_38,CoT_39,Final Answer_39,Instruction Violation_39,Prompt_File
0,GSM8K_test,Math,Richard lives in an apartment building with 15...,30,Step 1: Identify the total number of floors in...,30,"[(0, 0)]",Step 1: Identify the total number of floors in...,30,"[(0, 0)]",...,Step 1: Identify the given information - The b...,30,"[(0, 0)]",Step 1: Identify the given information - The b...,30,"[(0, 0)]",Step 1: Identify the total number of floors in...,30,"[(0, 0)]",few_CoT.json
1,GSM8K_test,Math,Shiela bought five cell phones for $150 each f...,255,Step 1: Identify the given information - Shiel...,255,"[(0, 0)]",Step 1: Identify the given information - Shiel...,255,"[(0, 0)]",...,Step 1: Identify the given information - Shiel...,255,"[(0, 0)]",Step 1: Identify the given information - Shiel...,255,"[(0, 0)]",Step 1: Identify the given information - Shiel...,255,"[(0, 0)]",few_CoT.json
2,GSM8K_test,Math,A robe takes 2 bolts of blue fiber and half th...,3,Step 1: Identify the given information - The r...,3,"[(0, 0)]",Step 1: Identify the given information - The r...,3,"[(0, 0)]",...,Step 1: Identify the given information - The r...,3,"[(0, 0)]",Step 1: Identify the given information - The r...,3,"[(0, 0)]",Step 1: Identify the given information - The r...,3,"[(0, 0)]",few_CoT.json
3,GSM8K_test,Math,Gunter is trying to count the jelly beans in a...,80,Step 1: Identify the guesses made by Gunter's ...,80,"[(0, 0)]",Step 1: Identify the guesses made by Gunter's ...,80,"[(0, 0)]",...,Step 1: Identify the guesses from Gunter's fri...,80,"[(0, 0)]",Step 1: Identify the guesses made by Gunter's ...,80,"[(0, 0)]",Step 1: Identify the guesses made by Gunter's ...,80,"[(0, 0)]",few_CoT.json
4,GSM8K_test,Math,Charlie wants to sell beeswax candles. For ev...,20,Step 1: Identify the given information - Each ...,20,"[(0, 0)]",Step 1: Identify the given information - Charl...,$20.00,"[(0, 0)]",...,Step 1: Identify the given information - Each ...,20,"[(0, 0)]",Step 1: Identify the given information - Charl...,$20.00,"[(0, 0)]",Step 1: Identify the given information - Charl...,20.00,"[(0, 0)]",few_CoT.json
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,GSM8K_test,Math,Tracy used a piece of wire 4 feet long to supp...,8,Step 1: Identify the given information - Total...,8,"[(0, 0)]",Step 1: Identify the given information - Total...,8,"[(0, 0)]",...,Step 1: Identify the given information - Total...,8,"[(0, 0)]",Step 1: Identify the given information - The t...,8,"[(0, 0)]",Step 1: Identify the given information - The t...,8,"[(0, 0)]",few_CoT.json
96,GSM8K_test,Math,Jean has 30 lollipops. Jean eats 2 of the loll...,14,Step 1: Identify the given information - Jean ...,14,"[(0, 0)]",Step 1: Identify the initial number of lollipo...,14,"[(0, 0)]",...,Step 1: Identify the initial number of lollipo...,14,"[(0, 0)]",Step 1: Identify the given information - Jean ...,14,"[(0, 0)]",Step 1: Identify the given information - Jean ...,14,"[(0, 0)]",few_CoT.json
97,GSM8K_test,Math,"Kelian has two recipes for preparing dishes, o...",60,Step 1: Identify the given information - The f...,60,"[(0, 0)]",Step 1: Identify the given information - The f...,60,"[(0, 0)]",...,Step 1: Identify the given information - The f...,60,"[(0, 0)]",Step 1: Identify the given information - The f...,60,"[(0, 0)]",Step 1: Identify the given information - The f...,60,"[(0, 0)]",few_CoT.json
98,GSM8K_test,Math,"According to its nutritional info, a bag of ch...",48,Step 1: Identify the given information - A bag...,48,"[(0, 0)]",Step 1: Identify the given information - A bag...,48g,"[(0, 0)]",...,Step 1: Identify the given information - A bag...,48,"[(0, 0)]",Step 1: Identify the given information - A bag...,48,"[(0, 0)]",Step 1: Identif

In [5]:
feature_li = ['LEN', 'QUA_IM', 'DIF_IV', 'SIM_COT_BIGRAM', 'SIM_COT_AGG', 'SIM_AC_BIGRAM', 'SIM_AC_AGG', 'SIM_INPUT', 'STEP_COUNT',  'STEP_COHERENCE'] 
# This includes total 10 features introduced in the paper; please see extract features for more details
data = extract_feature(df_test,feature_li)

jaccard with bigram time cost: 3.276160955429077s
jaccard with aggregation time cost: 31.41516089439392s


100%|██████████| 100/100 [00:07<00:00, 13.20it/s]


In [6]:
pd.DataFrame(data).head(5) # data is saved in json format

,id,Name,correct answer,CoT answers,Correctness,DIF_IV,MATH_TERM_DENSITY,LEN,SIM_INPUT,QUA_IM,SIM_COT_AGG,SIM_AC_BIGRAM,SIM_COT_BIGRAM,SIM_AC_AGG,IMPERATIVE_DENSITY,STEP_COUNT,STEP_COHERENCE,AVG_STEP_LENGTH
0,0,GSM8K_test,30,"[30.0, 30.0, 30.0, 30.0, 30.0, 30.0, 30.0, 30....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.056338028169014086, 0.06164383561643835, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, ...","[0.33333333333333337, 0.34615384615384615, 0.3...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.9545454545454546, 0.875, 0.63934426229...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.9545454545454546, 0.8333333333333334, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.028169014084507043, 0.0273972602739726, 0.0...","[6, 6, 6, 7, 6, 6, 8, 7, 7, 6, 5, 6, 6, 4, 5, ...","[0.3586818807790675, 0.3657331628303495, 0.343...","[20.666666666666668, 21.333333333333332, 22.66..."
1,1,GSM8K_test,255,"[255.0, 255.0, 255.0, 255.0, 255.0, 255.0, 255...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.029411764705882353, 0.027472527472527472, 0...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.2857142857142857, 0.25, 0.25, 0.30769230769...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.8615384615384616, 0.8028169014084507, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.8615384615384616, 0.8333333333333334, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.023529411764705882, 0.02197802197802198, 0....","[5, 5, 6, 5, 6, 6, 5, 6, 5, 6, 6, 5, 5, 6, 6, ...","[0.3033234126984127, 0.3433179723502304, 0.350...","[31.0, 33.4, 29.333333333333332, 30.8, 29.8333..."
2,2,GSM8K_test,3,"[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.009259259259259259, 0.009433962264150943, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.26415094339622647, 0.26415094339622647, 0.2...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.8, 0.7843137254901961, 0.8627450980392...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.8, 0.8695652173913043, 0.8888888888888...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.027777777777777776, 0.018867924528301886, 0...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.2896825396825397, 0.3627075351213282, 0.404...","[24.0, 23.5, 24.5, 23.75, 21.75, 24.0, 23.25, ..."
3,3,GSM8K_test,80,"[80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.016574585635359115, 0.0196078431372549, 0.0...","[1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, ...","[0.19999999999999996, 0.20779220779220775, 0.1...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.8524590163934427, 0.7746478873239436, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.8524590163934427, 0.7391304347826086, ...","[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.016574585635359115, 0.014705882352941176, 0...","[6, 5, 6, 4, 6, 6, 5, 5, 6, 4, 7, 5, 7, 5, 6, ...","[0.3032754010695187, 0.39781746031746035, 0.33...","[28.5, 38.4, 30.5, 50.5, 28.166666666666668, 3..."
4,4,GSM8K_test,20,"[20.0, $20.00, $20.00, $20.00, $20.00, 20.0, $...","[1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.02127659574468085, 0.023255813953488372, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.3835616438356164, 0.38961038961038963, 0.47...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.6883116883116883, 0.7283950617283951, ...","[0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, ...","[0.0, 0.6883116883116883, 0.7236842105263157, ...","[0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, ...","[0.0212765

In [7]:
df_processed = pd.DataFrame(data)

In [11]:
df_processed = calculate_SC_correctness(df_processed)

# Calculate Early Stopping Correctness with a specific window size
window_size = 5  # Define your window size
df_processed = calculate_ES_correctness(df_processed, window_size)

# Calculate Adaptive Consensus Correctness
df_processed = calculate_ASC_correctness(df_processed)

In [12]:
df_processed.head()#

,id,Name,correct answer,CoT answers,Correctness,DIF_IV,MATH_TERM_DENSITY,LEN,SIM_INPUT,QUA_IM,...,SIM_AC_AGG,IMPERATIVE_DENSITY,STEP_COUNT,STEP_COHERENCE,AVG_STEP_LENGTH,SC_correctness,ES_correctness,ES_steps,asc_correctness,asc_steps
0,0,GSM8K_test,30,"[30.0, 30.0, 30.0, 30.0, 30.0, 30.0, 30.0, 30....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.056338028169014086, 0.06164383561643835, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, ...","[0.33333333333333337, 0.34615384615384615, 0.3...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",...,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.028169014084507043, 0.0273972602739726, 0.0...","[6, 6, 6, 7, 6, 6, 8, 7, 7, 6, 5, 6, 6, 4, 5, ...","[0.3586818807790675, 0.3657331628303495, 0.343...","[20.666666666666668, 21.333333333333332, 22.66...",1,1,5,1,4
1,1,GSM8K_test,255,"[255.0, 255.0, 255.0, 255.0, 255.0, 255.0, 255...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.029411764705882353, 0.027472527472527472, 0...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.2857142857142857, 0.25, 0.25, 0.30769230769...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",...,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.023529411764705882, 0.02197802197802198, 0....","[5, 5, 6, 5, 6, 6, 5, 6, 5, 6, 6, 5, 5, 6, 6, ...","[0.3033234126984127, 0.3433179723502304, 0.350...","[31.0, 33.4, 29.333333333333332, 30.8, 29.8333...",1,1,5,1,4
2,2,GSM8K_test,3,"[3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, ...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.009259259259259259, 0.009433962264150943, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.26415094339622647, 0.26415094339622647, 0.2...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",...,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.027777777777777776, 0.018867924528301886, 0...","[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, ...","[0.2896825396825397, 0.3627075351213282, 0.404...","[24.0, 23.5, 24.5, 23.75, 21.75, 24.0, 23.25, ...",1,1,5,1,4
3,3,GSM8K_test,80,"[80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80.0, 80....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.016574585635359115, 0.0196078431372549, 0.0...","[1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, ...","[0.19999999999999996, 0.20779220779220775, 0.1...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",...,"[0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.016574585635359115, 0.014705882352941176, 0...","[6, 5, 6, 4, 6, 6, 5, 5, 6, 4, 7, 5, 7, 5, 6, ...","[0.3032754010695187, 0.39781746031746035, 0.33...","[28.5, 38.4, 30.5, 50.5, 28.166666666666668, 3...",1,1,5,1,4
4,4,GSM8K_test,20,"[20.0, $20.00, $20.00, $20.00, $20.00, 20.0, $...","[1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.02127659574468085, 0.023255813953488372, 0....","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.3835616438356164, 0.38961038961038963, 0.47...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",...,"[0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, ...","[0.02127659574468085, 0.023255813953488372, 0....","[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, ...","[0.3545866935483871, 0.3089693681125906, 0.298...","[34.6, 40.0, 34.0, 36.4, 39.2, 36.6, 36.8, 35....",0,0,13,0,12


In [ ]:
# TO DO 1: Demonstrate 1st way of doing without customize model; 
# 2: demonstrate how to do it with customized model;
# 4: Delete all unnecessary files in Gitrepo
# 5: Write doc on how to run the python file. (special explanations for feature extraction)

In [13]:
feature_li = ['LEN', 'QUA_IM', 'DIF_IV', 'SIM_COT_BIGRAM', 'SIM_AC_BIGRAM',  'SIM_INPUT', 'STEP_COUNT',  'STEP_COHERENCE'] # We can take less if for test
df_confidence_scores = customized_LR_model(df_processed, feature_li, report_auroc=False) # Note that we keep the test data only to aviod overfit

KeyError: 'Model'

In [25]:
df_llama_confidence_scores.head()

,id,Name,Model,correct answer,CoT answers,Correctness,MATH_TERM_DENSITY,STEP_COHERENCE,SIM_AC_AGG,SIM_COT_AGG,...,DIF_IV,QUA_IM,STEP_COUNT,SIM_AC_BIGRAM,SC_correctness,ES_correctness,ES_steps,asc_correctness,asc_steps,confidence_score
0,40,BigBench_easy,llama3,C,"[D, C, A, E, E, C, E, A, D, A, C, E, E, D, D, ...","[0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, ...","[0.5, 0.37209302325581395, 0.45033112582781454...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, ...",0,0,40,0,40,"[0.23638707935418551, 0.10582059028380454, 0.1..."
1,7,MathQA_challenge_test,llama3,c,"[E, C, E, E, E, E, E, E, C, C, D, D, E, B, E, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.012269938650306749, 0.02030456852791878, 0....","[0, 0, 0, 0, 0.32393939393939397, 0, 0, 0.2148...","[0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, ...","[0.5, 0.3652173913043478, 0.38, 0.254545454545...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, ...","[0, 0, 0, 1, 3, 0, 0, 5, 0, 0, 0, 0, 0, 0, 3, ...","[0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, ...",0,0,7,0,7,"[0.18175323341013105, 0.06319644962629163, 0.1..."
2,10,MathQA_dev,llama3,b,"[E, A, A, E, E, E, E, E, E, E, A, E, E, B, E, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.005050505050505051, 0.011764705882352941, 0...","[0.2361111111111111, 0.22580645161290322, 0, 0...","[0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, ...","[0.5, 0.6931818181818181, 0.6274509803921569, ...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[2, 2, 0, 0, 0, 2, 0, 0, 0, 2, 2, 2, 0, 2, 0, ...","[0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, ...",0,0,8,0,10,"[0.18299950629311443, 0.41321052181956053, 0.4..."
3,44,MathQA_challenge_test,llama3,a,"[E, E, E, D, E, E, E, E, E, E, E, E, E, E, E, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.011764705882352941, 0.011494252873563218, 0...","[0, 0, 0, 0, 0, 0, 0, 0, 0.32205919503079744, ...","[0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[0.5, 0.38497652582159625, 0.296137339055794, ...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, ...","[0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...",0,0,9,0,7,"[0.17491185964273384, 0.18835863334431088, 0.1..."
4,33,BigBench_easy,llama3,D,"[B, E, C, A, C, C, C, E, A, E, C, A, E, E, C, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, ...","[0.5, 0.45112781954887216, 0.3549382716049383,...",...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, ...",0,0,40,0,40,"[0.24724127099954366, 0.18596011704801862, 0.0..."


In [26]:
N = 5
threshold = 0.5

# Applying early stopping mechanism
df_final = CS_early_stopping(df=df_llama_confidence_scores, threshold=threshold, N=N)

SC_ACC : 0.2
ES_ACC : 0.2
CS_ACC : 0.3333333333333333
SC_Avg_Steps : 40
ES_Avg_Steps : 21.6
CS_Avg_Steps : 34.266666666666666
ASC_Avg_Steps : 18.066666666666666
ASC_ACC : 0.2


# How to Run the Code the get CoTs?